# Example 12.6.3 / 1: Solve a two-stage order

Companion notebook for **12.6.3 Two-Stage Recourse and Nonanticipativity** in the AMPL lab Documentation, section "Solve a two-stage order". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve a two-stage order

A one-item order shows the same structure on numbers. `Order` is the here-and-now decision and
`Sell[s]` the recourse after demand `s` is revealed. Units cost 4 and sell for 10; demand is
40, 100 or 130 with probabilities 0.3, 0.4 and 0.3. After the first solve, `ampl.var["Order"].fix(91)`
evaluates the expected-value order (mean demand `0.3*40 + 0.4*100 + 0.3*130 = 91`) on the same
scenarios.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set SCENARIOS;                          # demand outcomes
param probability {SCENARIOS} >= 0;     # scenario weight
param demand {SCENARIOS} >= 0;          # units
param unit_cost >= 0;                   # dollars per unit ordered
param price >= 0;                       # dollars per unit sold

# First stage: one order before demand is known.
var Order >= 0;

# Second stage: sales adapt to each revealed demand.
var Sell {SCENARIOS} >= 0;

maximize ExpectedProfit:                # order cost now, expected sales revenue later
    - unit_cost * Order
  + sum {s in SCENARIOS} probability[s] * price * Sell[s];

# Sales cannot exceed the order or the scenario demand.
subject to SellFromOrder {s in SCENARIOS}: Sell[s] <= Order;
# sales capped by the revealed demand
subject to SellToDemand {s in SCENARIOS}: Sell[s] <= demand[s];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["SCENARIOS"] = ["low", "mid", "high"]
ampl.param["probability"] = {"low": 0.3, "mid": 0.4, "high": 0.3}
ampl.param["demand"] = {"low": 40, "mid": 100, "high": 130}
ampl.param["unit_cost"] = 4
ampl.param["price"] = 10

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.var["Order"].value(), ampl.obj["ExpectedProfit"].value())
print(ampl.var["Sell"].to_dict())

ampl.var["Order"].fix(91)       # the expected-value order
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result, ampl.obj["ExpectedProfit"].value())

solved 100.0 420.0
{'high': 100, 'low': 40, 'mid': 100}
solved 393.0


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`; `fix Order := 91` replaces the amplpy `fix`:

In [6]:
%%writefile data.dat
set SCENARIOS := low mid high;
param probability := low 0.3  mid 0.4  high 0.3;
param demand := low 40  mid 100  high 130;
param unit_cost := 4;
param price := 10;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Order, ExpectedProfit;
display Sell;
fix Order := 91;
solve;
display ExpectedProfit;

HiGHS 1.14.0: optimal solution; objective 420
1 simplex iterations
0 barrier iterations
Order = 100
ExpectedProfit = 420

Sell [*] :=
high  100
 low   40
 mid  100
;

HiGHS 1.14.0: optimal solution; objective 393
0 simplex iterations
0 barrier iterations
ExpectedProfit = 393

